In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

mat_1 = np.ones((5, 5))

# Exercise 4

# perform slicing
mat_1[1:4, 1:4] = 0
print(mat_1)

# task 2
np.random.seed(42)
random_data = np.random.randn(100, 3)

# print(random_data)
normalized = (random_data - random_data.mean(axis=0)) / random_data.std(axis=0)
print(normalized)

# task 3
X = np.random.randn(50, 3)

true_theta = np.array([2.5, -1.2, 3.7])

y = X @ true_theta + np.random.randn(50) * 0.1

# Normal equation
theta_hat = np.linalg.inv(X.T @ X) @ X.T @ y

print("\nTask 3:")
print("Estimated coefficients:")
print(theta_hat)

print("\nTrue coefficients:")
print(true_theta)



In [ ]:
# Exercise 5
"""
Exercise 5: Pandas Data Analysis
Analyze a dataset of student performance.
"""


# Create sample dataset
np.random.seed(42)
n_students = 200

data = {
    'student_id': range(1000, 1000 + n_students),
    'major': np.random.choice(['CS', 'Math', 'Physics', 'Biology'], n_students),
    'year': np.random.choice([1, 2, 3, 4], n_students),
    'exam_score': np.random.normal(75, 10, n_students).clip(0, 100),
    'assignments_completed': np.random.randint(0, 11, n_students),
    'hours_studied': np.random.normal(15, 5, n_students).clip(1, 40)
}

df = pd.DataFrame(data)

# Introduce some NaN values
df.loc[np.random.choice(n_students, 10), 'exam_score'] = np.nan
df.loc[np.random.choice(n_students, 5), 'hours_studied'] = np.nan

# task 1

# display basic information about the dataset
print(df.describe()) 

# identify and count missing values
df.isnull().sum()

# fill missing exam_score with the mean score for the student's major
df['exam_score'] = df['exam_score'].fillna(
    df.groupby('major')['exam_score'].transform('mean')
)

# fill missing hours_studied with the median for the student's year
df['hours_studied'] = df['hours_studied'].fillna(
    df.groupby('year')['hours_studied'].transform('median'))

print()

# task 2

# calculate and display the average exam_score by major
avg_table = df.groupby('major')['exam_score'].agg(['mean']).rename(columns={'mean': 'average'})
print(avg_table)

# find the major with the highest average exam_score
avg_by_major = df.groupby('major')['exam_score'].mean()

print()

top_major = avg_by_major.idxmax()
top_score = avg_by_major.max()

print("Top Major:", top_major)
print(f"Average Score: {top_score:.2f}")

print()

# calculate the correlation between hours_studied and exam_score
corrl = df['hours_studied'].corr(df['exam_score'])
print(f"The correlation between hours_studied and exam_score: {corrl}")

col_category = [df['exam_score'] > 90,
    (df['exam_score'] >= 80) & (df['exam_score'] <= 90),
    (df['exam_score'] >= 70) & (df['exam_score'] < 80),
    df['exam_score'] < 70
]

choices = ['Excellent', 'Good', 'Average', 'Needs Improvement']

df['performance'] = np.select(col_category, choices, default ='Unknown')

print()

# task 3
summary = df.groupby(['major', 'year']).agg(
    number_of_students=('student_id', 'count'),
    average_exam_score=('exam_score', 'mean'),
    average_hours_studied=('hours_studied', 'mean')
)

print(summary)

# identify top 5 students based on exam_score (handle ties appropriately) 
top_score_threshold = df['exam_score'].nlargest(5).min()

top_students = df[df['exam_score'] >= top_score_threshold]

print(top_students)

print()

# create a pivot table showing average exam_score by major (rows) and year (columns)
pivot = df.pivot_table(
    values = 'exam_score',
    index = 'major',
    columns = 'year',
    aggfunc = 'mean'
)

print(pivot)


In [ ]:
"""
Exercise 6: Data Visualization
Create meaningful visualizations using the dataset from Exercise 5.
"""

# task 1

# create a figure with 2 subplots side by side
# plt.figure(figsize=(14,6))

plt.figure(figsize=(14,6))


plt.subplot(1, 2, 1)
sns.histplot(df['exam_score'], kde=True, bins=20, color='pink') # type: ignore

plt.title('Distribution of Exam Scores')
plt.xlabel('Exam Score')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.boxplot(x='major', y='exam_score', data=df, color="blue")

plt.title('Exam Scores by Major')
plt.xlabel('Major')
plt.ylabel('Exam Score')
plt.subplot(1, 2, 2)
sns.boxplot(x='major', y='exam_score', data=df, color="lightgreen")

plt.title('Exam Scores by Major')
plt.xlabel('Major')
plt.ylabel('Exam Score')

# task 2

plt.figure(figsize=(10, 6))

sns.set_theme(style="whitegrid")

# Scatter plot with regression line + color by major
sns.lmplot(
    data=df,
    x='hours_studied',
    y='exam_score',
    hue='major',
    height=6,
    aspect=1.5,
    scatter_kws={'alpha': 0.7}
)

plt.title('Relationship between Hours Studied and Exam Score')
plt.xlabel('Hours Studied')
plt.ylabel('Exam Score')

plt.show()

# task 3

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))


# 1. Bar chart: Avg exam score by major
avg_major = df.groupby('major')['exam_score'].mean()

avg_major.plot(kind='bar', ax=axes[0, 0], color='teal')
axes[0, 0].set_title('Average Exam Score by Major')
axes[0, 0].set_xlabel('Major')
axes[0, 0].set_ylabel('Average Exam Score')

# 2. Count plot: Students by year
sns.countplot(data=df, x='year', ax=axes[0, 1], palette='Set2')

axes[0, 1].set_title('Number of Students by Year')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Count')

# 3. Heatmap: Correlation matrix
corr = df[['exam_score', 'hours_studied', 'assignments_completed']].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', ax=axes[1, 0])

axes[1, 0].set_title('Correlation Matrix')

# 4. Violin plot: Exam score by performance
sns.violinplot(
    data=df,
    x='performance',
    y='exam_score',
    ax=axes[1, 1],
    palette='Pastel1'
)

axes[1, 1].set_title('Exam Score Distribution by Performance')
axes[1, 1].set_xlabel('Performance Category')
axes[1, 1].set_ylabel('Exam Score')

# Final layout adjustments
plt.tight_layout()
plt.show()


In [78]:
"""
Exercise 7: Integration Challenge
Combine NumPy, Pandas, and Matplotlib to solve a mini data science problem.
"""

max_freq = df['purchase_frequency'].max()

df = df.assign(
    churn_risk=1 - (df['purchase_frequency'] / max_freq)
)

df['CLV'] = df['purchase_frequency'] * df['avg_purchase_value'] * (1 + df['churn_risk'])


##Age Grouping

def group_age(x):
    if 18 <= x <= 25:
        return "18-25"
    elif 26 <= x <= 35:
        return "26-35"
    elif 36 <= x <= 50:
        return "36-50"
    else:
        return "51-70"

df['age_group'] = df['age'].apply(group_age)

##Group Analysis
summary = df.groupby('age_group').agg(
    customers=('age', 'size'),
    avg_income=('income', 'mean'),
    avg_clv=('CLV', 'mean'),
    total_clv=('CLV', 'sum')
).reset_index()

summary

KeyError: 'purchase_frequency'